<a href="https://colab.research.google.com/github/ShashikanthKungulwar/DL/blob/main/CNN_1st.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# from torchvision.utils.data import DataLoader

In [2]:
transform = transforms.ToTensor()

In [3]:
train_data = datasets.MNIST(root="/cnn_data",download=True,transform=transform,train=True)
test_data = datasets.MNIST(root="/cnn_data",download=True,transform=transform,train=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 129MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 45.7MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 73.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.63MB/s]


In [4]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: /cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [5]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [6]:
pwd

'/content'

In [7]:
ls


sample_data/


In [8]:
cd ..


/


In [9]:
ls


bin@       datalab/  kaggle/  libx32@  proc/               run/   tmp/
boot/      dev/      lib@     media/   python-apt/         sbin@  tools/
cnn_data/  etc/      lib32@   mnt/     python-apt.tar.xz*  srv/   usr/
content/   home/     lib64@   opt/     root/               sys/   var/


In [10]:
train_data_loader = DataLoader(train_data)

In [11]:
train_loader = DataLoader(train_data,batch_size=10,shuffle=True)
test_loader = DataLoader(test_data,shuffle=False,batch_size=10)

In [12]:
type(train_loader)

torch.utils.data.dataloader.DataLoader

In [13]:
type(train_data)

torchvision.datasets.mnist.MNIST

In [14]:
train_data[0][0].shape # depth cols and rows

torch.Size([1, 28, 28])

In [15]:
conv1 = nn.Conv2d(1,6,3,1)

In [16]:
F.relu(conv1(train_loader))

TypeError: conv2d() received an invalid combination of arguments - got (DataLoader, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!DataLoader!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!DataLoader!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)


In [18]:
class CNNModel(nn.Module):
  def __init__(self,in_depth = 1,out_depth=10) :
    super().__init__()
    self.conv1 = nn.Conv2d(in_depth,6,3,1,padding="same")
    self.conv2 = nn.Conv2d(6,9,3,1,padding = "same")
    self.conv3 = nn.Conv2d(9,10,3,1,padding = "same")
    # self.conv4 = nn.conv2d(10,)
    self.fc1 = nn.Linear(3*3*10,70)
    self.fc2 = nn.Linear(70,30)
    self.fc3 = nn.Linear(30,out_depth)

  def forward(self,x):
    x = F.relu(self.conv1(x))
    x = F.max_pool2d(x,2,2)
    x = F.relu(self.conv2(x))
    x = F.max_pool2d(x,2,2)
    x = F.relu(self.conv3(x))
    x = F.max_pool2d(x,2,2)
    x = x.view(-1,x.shape[1]*x.shape[2]*x.shape[3])

    x= F.relu(self.fc1(x))
    x= F.relu(self.fc2(x))
    x= F.log_softmax(self.fc3(x),dim =1)

    return x



In [19]:
torch.manual_seed(41)
model = CNNModel()

In [20]:
model

CNNModel(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (conv2): Conv2d(6, 9, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (conv3): Conv2d(9, 10, kernel_size=(3, 3), stride=(1, 1), padding=same)
  (fc1): Linear(in_features=90, out_features=70, bias=True)
  (fc2): Linear(in_features=70, out_features=30, bias=True)
  (fc3): Linear(in_features=30, out_features=10, bias=True)
)

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [22]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [23]:
epochs = 5
for epoch in range(epochs):

  model.train()
  for b,(x_train,y_train) in enumerate(train_loader):
    y_pred = model.forward(x_train)

    loss = criterion(y_pred,y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    if(b%600 == 0):
      print(f"epoch : {epoch} and batch : {b} and loss:{loss.item()}")



  model.eval()
  total_loss = 0
  total = 0
  correct = 0
  with torch.no_grad():
    for b,(x_test,y_test) in enumerate(test_loader):
      y_pred = model.forward(x_test)

      loss = criterion(y_pred,y_test)
      total_loss += loss.item() * y_test.size(0)
      total += y_test.size(0)
      # _,predicted = torch.max(y_pred.data,1)
      predicted = torch.max(y_pred,dim = 1)[1]
      correct += (predicted == y_test).sum().item()

      # print(f"epoch : {epoch} and loss:{loss.item()}")

  print(f"epoch : {epoch} and loss : {total_loss/total} and acc : {correct/total}")




epoch : 0 and batch : 0 and loss:2.348390579223633
epoch : 0 and batch : 600 and loss:0.671683669090271
epoch : 0 and batch : 1200 and loss:0.11230949312448502
epoch : 0 and batch : 1800 and loss:0.20084360241889954
epoch : 0 and batch : 2400 and loss:0.22600583732128143
epoch : 0 and batch : 3000 and loss:0.010198374278843403
epoch : 0 and batch : 3600 and loss:0.16228793561458588
epoch : 0 and batch : 4200 and loss:0.4354383945465088
epoch : 0 and batch : 4800 and loss:0.26719778776168823
epoch : 0 and batch : 5400 and loss:0.0175869669765234
epoch : 0 and loss : 0.10538543490089068 and acc : 0.9676
epoch : 1 and batch : 0 and loss:0.007127438671886921
epoch : 1 and batch : 600 and loss:0.10298652946949005
epoch : 1 and batch : 1200 and loss:0.0020088304299861193
epoch : 1 and batch : 1800 and loss:0.13493207097053528
epoch : 1 and batch : 2400 and loss:0.2037728726863861
epoch : 1 and batch : 3000 and loss:0.0026406750548630953
epoch : 1 and batch : 3600 and loss:0.00965745653957128